# 06 — Preprocessing (eye-tracking)

1. **Janelas de face** — detecta por visão computacional quando há face na tela no vídeo padrão
   (T30) e alinha as 50 janelas em cada gravação pelo fim.
2. **Recorte** — mantém só as amostras de gaze dentro das janelas (`*_ON.csv`).
3. **Critério de inclusão** — 50 janelas válidas + critério técnico de `src/qualidade_sujeitos.py`.
4. **Gaze com emoção** — uma tabela única (`gaze_com_emocoes.csv`) com a emoção de cada amostra.

Entrada: `data/raw/TABELASCSV/` (do `01_Structuring_et`). Saídas em `data/interim/` e
`data/processed/gaze_com_emocoes.csv`.

In [1]:
import os, glob, sys
from pathlib import Path
import numpy as np
import pandas as pd
import cv2

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

DIR_RAW       = RAIZ / "data" / "raw"
DIR_INTERIM   = RAIZ / "data" / "interim"
DIR_PROCESSED = RAIZ / "data" / "processed"
DIR_INTERIM.mkdir(parents=True, exist_ok=True)
DIR_PROCESSED.mkdir(parents=True, exist_ok=True)

PASTA_XLSX         = DIR_RAW / "TABELAS_COORDENADAS_EYE-TRACKING_FACES_TODOS_OS_TESTES"
PASTA_CSV          = DIR_RAW / "TABELASCSV"
VIDEO_PADRAO       = PASTA_XLSX / "VIDEO T30 TEA GRAVE.mp4"   # vídeo molde (T30)
DIR_JANELAS_ON     = DIR_INTERIM / "janelas_on"
JANELAS_ON_TODOS   = DIR_INTERIM / "janelas_on_todos.csv"
JANELAS_ON_VALIDOS = DIR_INTERIM / "janelas_on_validos.csv"
PASTA_FILTRADOS    = DIR_INTERIM / "originais_filtrados"
ORIGINAIS_TODOS    = DIR_INTERIM / "originais_todos.csv"
GAZE_COM_EMOCOES   = DIR_PROCESSED / "gaze_com_emocoes.csv"
print("Raiz do projeto:", RAIZ)

Raiz do projeto: /Users/marcosgomes/Developer/ISD/eye_tracking_faces


## 1. Detecção das janelas de face por vídeo

Classifica cada frame do vídeo padrão em face / cruz de fixação / branco e extrai as 50 janelas
de face. Como o tempo de calibração no início varia entre participantes, as janelas são
ancoradas no **fim** de cada gravação. Emoção por posição: feliz → neutro → raiva.

In [2]:
# ═══════════════════════════════════════════════════════════════════
# PARÂMETROS DE DETECÇÃO — edite aqui se o número de janelas estiver errado
# ═══════════════════════════════════════════════════════════════════

SEQ_EMOCOES = ["feliz", "neutro", "raiva"]   # ordem cronológica fixa

WHITE_V_THR  = 200    # brilho médio acima disso → BRANCO (não mexer)

# Saturação central (excluindo círculo vermelho e pontos verdes de eyetracking)
FACE_SAT_THR   = 15    # sat_mean ≥ isso → claramente FACE
# Saturação do canto inferior esquerdo (segundo discriminador)
# CRUZ_FIX: fundo cinza → sat_canto ≈ 0-5
# FACE pálida: chin/pescoço/roupa → sat_canto ≥ 8-20
CORNER_SAT_THR = 8     # sat canto ≥ isso → FACE mesmo com sat_mean baixo
CORNER_H_FRAC  = 0.25  # fração da altura: examina o bottom 25% do frame
CORNER_W_FRAC  = 0.25  # fração da largura: examina o left  25% do frame

# Janelas de face reais têm ~3s; flashes espúrios têm < 1s.
# Diminuir se alguma face real for muito curta; aumentar para filtrar mais ruído.
MIN_FACE_DUR_S = 1.5   # descarta janelas FACE com duração abaixo disso

MIN_RUN_FRAMES = 2     # frames mínimos para confirmar mudança de estado
# ═══════════════════════════════════════════════════════════════════


In [3]:
import cv2
import numpy as np


def metricas_frame(bgr):
    """Retorna (v_mean, sat_mean, sat_canto)."""
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    h, w = hsv.shape[:2]

    v_mean = float(hsv[..., 2].mean())

    # Saturação central excluindo verde (eyetracking) e vermelho (cruz)
    cy0, cy1 = int(h * 0.15), int(h * 0.85)
    cx0, cx1 = int(w * 0.20), int(w * 0.80)
    central = hsv[cy0:cy1, cx0:cx1]
    H_c, S_c = central[..., 0], central[..., 1]
    mask_green = (H_c >= 70) & (H_c <= 100) & (S_c > 40)
    mask_red_c = ((H_c <= 10) | (H_c >= 170)) & (S_c > 80)
    valid = ~(mask_green | mask_red_c)
    sat_mean = float(central[..., 1][valid].mean()) if valid.any() else 0.0

    # Saturação do canto inferior esquerdo (longe do círculo e dos pontos)
    y0_c = int(h * (1 - CORNER_H_FRAC))
    x1_c = int(w * CORNER_W_FRAC)
    sat_canto = float(hsv[y0_c:, :x1_c, 1].mean())

    return v_mean, sat_mean, sat_canto


def classificar(v_mean, sat_mean, sat_canto):
    if v_mean >= WHITE_V_THR:
        return "BRANCO"
    if sat_mean >= FACE_SAT_THR:
        return "FACE"
    if sat_canto >= CORNER_SAT_THR:
        return "FACE"
    return "CRUZ_FIX"


def suavizar(estados, min_run):
    estados = list(estados)
    n = len(estados)
    i = 1
    while i < n:
        if estados[i] != estados[i - 1]:
            j = i
            while j < n and estados[j] == estados[i]:
                j += 1
            if (j - i) < min_run:
                for k in range(i, j):
                    estados[k] = estados[i - 1]
            i = j
        else:
            i += 1
    return estados


def detecta_janelas_face(video_path):
    """Retorna (duracao_video_s, [(ini_s, fim_s), ...]) das janelas FACE."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Não consegui abrir o vídeo: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS)

    estados_raw = []
    sat_vals, canto_vals = [], []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame_s = cv2.resize(frame, (320, 240), interpolation=cv2.INTER_AREA)
        v, s, c = metricas_frame(frame_s)
        estados_raw.append(classificar(v, s, c))
        sat_vals.append(s)
        canto_vals.append(c)
    cap.release()

    estados = suavizar(estados_raw, MIN_RUN_FRAMES)
    n     = len(estados)
    dur_s = n / fps

    # Coleta todos os segmentos FACE
    segmentos = []
    k = 0
    while k < n:
        if estados[k] == "FACE" and (k == 0 or estados[k - 1] != "FACE"):
            ini = k
            while k < n and estados[k] == "FACE":
                k += 1
            fim = k
            dur = (fim - ini) / fps
            segmentos.append((ini, fim, dur))
        else:
            k += 1

    # Filtra janelas curtas demais (artefatos de transição)
    janelas_raw = segmentos
    janelas = [(ini/fps, fim/fps) for ini, fim, dur in janelas_raw if dur >= MIN_FACE_DUR_S]
    descartadas = [(ini/fps, fim/fps, dur) for ini, fim, dur in janelas_raw if dur < MIN_FACE_DUR_S]

    if descartadas:
        print(f"  Descartadas {len(descartadas)} janelas curtas (< {MIN_FACE_DUR_S}s):")
        for i_s, f_s, d in descartadas:
            print(f"    {i_s:.2f}s → {f_s:.2f}s  ({d:.2f}s)")

    for idx, (ini_s, fim_s) in enumerate(janelas):
        ini, fim = int(ini_s * fps), int(fim_s * fps)
        sats  = sat_vals[ini:fim]
        cantos = canto_vals[ini:fim]
        print(f"  janela {idx+1:>2}: {ini_s:6.2f}s → {fim_s:6.2f}s"
              f"  sat_med={sorted(sats)[len(sats)//2]:.1f}"
              f"  canto_med={sorted(cantos)[len(cantos)//2]:.1f}")

    if len(janelas) != 50:
        print(f"\n  ⚠ {len(janelas)} janelas (esperado 50). Ajuste MIN_FACE_DUR_S ou CORNER_SAT_THR.")
    else:
        print(f"\n  ✓ {len(janelas)} janelas detectadas")

    return dur_s, janelas


In [4]:
# ── Detecta as janelas ON no vídeo padrão (T30) e alinha em cada participante pelo FIM da gravação ──
SEQ_EMOCOES_PIPE = SEQ_EMOCOES

# ----- METODO VIDEO: detecta no T30 usando cv2 (preserva ritmo real) -----
print(f"Detectando janelas em {VIDEO_PADRAO.name}")
_dur_video_s, _janelas = detecta_janelas_face(str(VIDEO_PADRAO))
print(f"Duração: {_dur_video_s:.2f}s | {len(_janelas)} janelas FACE detectadas\n")
# molde: distância do fim + emoção por posição
_molde = []
for _idx, (_ini_s, _fim_s) in enumerate(_janelas):
    _molde.append({
        "n_face": _idx + 1,
        "emocao": SEQ_EMOCOES_PIPE[_idx % len(SEQ_EMOCOES_PIPE)],
        "ini_do_fim_s": _dur_video_s - _ini_s,
        "fim_do_fim_s": _dur_video_s - _fim_s,
    })

DIR_JANELAS_ON.mkdir(parents=True, exist_ok=True)
_todos = []
for _csv in sorted(PASTA_CSV.glob("*.csv")):
    _d = pd.read_csv(_csv)
    if "Time (ms)" not in _d.columns:
        continue
    _t_fim = float(pd.to_numeric(_d["Time (ms)"], errors="coerce").max())
    _linhas = []
    for _m in _molde:
        _ini = _t_fim - _m["ini_do_fim_s"] * 1000.0
        _fim = _t_fim - _m["fim_do_fim_s"] * 1000.0
        _linhas.append({
            "Arquivo": _csv.name, "n_face": _m["n_face"], "emocao": _m["emocao"],
            "on_inicio_ms": round(_ini, 1), "on_fim_ms": round(_fim, 1),
            "duracao_ms": round(_fim - _ini, 1),
        })
    _df_ind = pd.DataFrame(_linhas)
    _df_ind = _df_ind[_df_ind["on_inicio_ms"] >= 0].reset_index(drop=True)
    _df_ind.to_csv(DIR_JANELAS_ON / f"{_csv.stem}_janelas_on.csv", index=False)
    _todos.append(_df_ind)
    print(f"  {_csv.stem:<26} fim={_t_fim/1000:7.2f}s | {len(_df_ind)} janelas")


if _todos:
    _df_todos = pd.concat(_todos, ignore_index=True)
    _df_todos.to_csv(JANELAS_ON_TODOS, index=False)
    print(f"\nConsolidado salvo: {JANELAS_ON_TODOS.name} ({len(_df_todos)} linhas)")


Detectando janelas em VIDEO T30 TEA GRAVE.mp4


  Descartadas 1 janelas curtas (< 1.5s):
    46.68s → 47.12s  (0.44s)
  janela  1:   2.76s →   5.52s  sat_med=29.8  canto_med=8.0
  janela  2:   6.40s →   9.12s  sat_med=16.8  canto_med=8.0
  janela  3:  10.12s →  13.04s  sat_med=21.7  canto_med=8.0
  janela  4:  14.00s →  16.76s  sat_med=44.6  canto_med=8.0
  janela  5:  17.68s →  20.40s  sat_med=25.5  canto_med=8.0
  janela  6:  21.36s →  24.08s  sat_med=39.2  canto_med=8.0
  janela  7:  25.08s →  27.68s  sat_med=18.1  canto_med=8.0
  janela  8:  28.64s →  31.48s  sat_med=21.9  canto_med=8.0
  janela  9:  32.36s →  35.28s  sat_med=22.8  canto_med=8.0
  janela 10:  36.24s →  38.92s  sat_med=37.0  canto_med=8.0
  janela 11:  39.80s →  42.56s  sat_med=16.4  canto_med=8.0
  janela 12:  43.48s →  46.32s  sat_med=27.2  canto_med=8.0
  janela 13:  47.32s →  50.00s  sat_med=35.2  canto_med=8.0
  janela 14:  51.20s →  53.72s  sat_med=12.9  canto_med=8.0
  janela 15:  54.68s →  57.32s  sat_med=38.8  canto_med=8.0
  janela 16:  58.28s →  61.12s

  T22 TEA MODERADO           fim= 203.90s | 50 janelas
  T25 TEA LEVE               fim= 203.85s | 50 janelas
  T26 TEA GRAVE              fim= 203.61s | 50 janelas
  T27 TEA LEVE               fim= 203.73s | 50 janelas
  T28 TEA GRAVE              fim= 203.91s | 50 janelas
  T29 TEA LEVE               fim= 203.81s | 50 janelas
  T30 TEA GRAVE              fim= 203.80s | 50 janelas
  T32 TEA LEVE               fim= 203.71s | 50 janelas
  T34 CONTROLE               fim= 203.86s | 50 janelas
  T35 CONTROLE               fim= 203.88s | 50 janelas
  T36 CONTROLE               fim= 203.85s | 50 janelas
  T37 CONTROLE               fim= 203.91s | 50 janelas


  T38 CONTROLE               fim= 203.60s | 50 janelas
  T39 CONTROLE               fim= 203.93s | 50 janelas
  T40 TEA GRAVE              fim= 203.85s | 50 janelas
  T41 CONTROLE               fim= 203.86s | 50 janelas
  T42 CONTROLE               fim= 205.99s | 50 janelas
  T43 CONTROLE               fim= 203.88s | 50 janelas
  T44 CONTROLE               fim= 203.88s | 50 janelas
  T46 TEA LEVE               fim= 203.78s | 50 janelas
  T47 CONTROLE               fim= 203.00s | 50 janelas
  T48 CONTROLE               fim= 203.86s | 50 janelas
  T49 CONTROLE               fim= 203.85s | 50 janelas


  T50 CONTROLE               fim= 203.86s | 50 janelas
  T51 CONTROLE               fim= 203.90s | 50 janelas
  T54 CONTROLE               fim= 203.95s | 50 janelas
  T55 CONTROLE               fim= 203.85s | 50 janelas
  T56 CONTROLE               fim= 203.93s | 50 janelas
  T57 CONTROLE               fim= 203.88s | 50 janelas
  T58 CONTROLE               fim= 203.90s | 50 janelas
  T59 CONTROLE               fim= 203.90s | 50 janelas
  T60 CONTROLE               fim= 203.95s | 50 janelas
  T61 CONTROLE               fim= 203.90s | 50 janelas

Consolidado salvo: janelas_on_todos.csv (2100 linhas)


## 2. Recorte do gaze nas janelas de face (`*_ON.csv`) e consolidação

In [5]:
# ── Gera os _ON.csv (todos os pontos com face na tela) ───────────
# Agora usa o arquivo de janelas detectadas por vídeo (janelas_on_todos.csv),
# que substitui a antiga auditoria. As janelas já vêm alinhadas pelo fim e
# na mesma escala de 'Time (ms)' dos dados, então é só recortar.
pasta_entrada = str(PASTA_CSV)
pasta_saida   = str(PASTA_FILTRADOS)
arquivo_janelas = str(JANELAS_ON_TODOS)

os.makedirs(pasta_saida, exist_ok=True)

# 1. Carregar as janelas ON (uma linha por face, com on_inicio_ms / on_fim_ms)
df_jan = pd.read_csv(arquivo_janelas)

# Agrupar as janelas por arquivo
janelas_por_arquivo = {
    nome: grupo[['on_inicio_ms', 'on_fim_ms']].values.tolist()
    for nome, grupo in df_jan.groupby('Arquivo')
}

# 2. Processar os arquivos de dados
csv_files = glob.glob(os.path.join(pasta_entrada, "*.csv"))

for caminho_arquivo in csv_files:
    nome_arquivo = os.path.basename(caminho_arquivo)

    if nome_arquivo not in janelas_por_arquivo:
        continue

    df = pd.read_csv(caminho_arquivo)

    valores_on = []
    # Recorta TODAS as janelas de face (não só as 3 primeiras)
    for inicio_on, fim_on in janelas_por_arquivo[nome_arquivo]:
        filtro = df[(df["Time (ms)"] >= inicio_on) & (df["Time (ms)"] < fim_on)]
        if not filtro.empty:
            valores_on.append(filtro)

    nome_saida = os.path.splitext(nome_arquivo)[0] + "_ON.csv"
    caminho_final = os.path.join(pasta_saida, nome_saida)

    if valores_on:
        df_on = pd.concat(valores_on)
        df_on.to_csv(caminho_final, index=False)
        print(f"[{nome_arquivo}] Salvo! Linhas ON: {len(df_on)}")
    else:
        pd.DataFrame(columns=df.columns).to_csv(caminho_final, index=False)
        print(f"[{nome_arquivo}] 0 linhas ON (arquivo vazio salvo)")


[T54 CONTROLE.csv] Salvo! Linhas ON: 7439
[T41 CONTROLE.csv] Salvo! Linhas ON: 6280
[T21 TEA MODERADO.csv] Salvo! Linhas ON: 4186


[T37 CONTROLE.csv] Salvo! Linhas ON: 8087
[T47 CONTROLE.csv] Salvo! Linhas ON: 8123


[T12 CONTROLE.csv] Salvo! Linhas ON: 8215
[T191 TEA MODERADO.csv] Salvo! Linhas ON: 7873


[T55 CONTROLE.csv] Salvo! Linhas ON: 7590
[T36 CONTROLE.csv] Salvo! Linhas ON: 7976


[T26 TEA GRAVE.csv] Salvo! Linhas ON: 7419
[T20 TEA MODERADO.csv] Salvo! Linhas ON: 6703
[T28 TEA GRAVE.csv] Salvo! Linhas ON: 1984


...

[T30 TEA GRAVE.csv] Salvo! Linhas ON: 6997
[T22 TEA MODERADO.csv] Salvo! Linhas ON: 7358


[T51 CONTROLE.csv] Salvo! Linhas ON: 8164
[T18 TEA MODERADO.csv] Salvo! Linhas ON: 7707


[T56 CONTROLE.csv] Salvo! Linhas ON: 8065
[T43 CONTROLE.csv] Salvo! Linhas ON: 7600


[T44 CONTROLE.csv] Salvo! Linhas ON: 7717
[T60 CONTROLE.csv] Salvo! Linhas ON: 8079


[T35 CONTROLE.csv] Salvo! Linhas ON: 7095


In [6]:
import pandas as pd
import glob
import os

# 1. Buscar os arquivos (lembre-se de ajustar o "*.csv" se necessário para não pegar lixo)
arquivos = glob.glob(os.path.join(str(PASTA_FILTRADOS), "*.csv"))

# Aviso caso a pasta esteja vazia: os _ON.csv são gerados pelas células
# de filtragem acima. Rode o notebook de cima para baixo antes desta célula.
if not arquivos:
    raise FileNotFoundError(
        f"Nenhum _ON.csv encontrado em {PASTA_FILTRADOS}. "
        "Rode as células de filtragem (acima) antes desta."
    )

# Lista que vai guardar as tabelas temporariamente
lista_dataframes = []

for arquivo in arquivos:
    # Lemos o arquivo atual da repetição
    df_temp = pd.read_csv(arquivo)   # _ON.csv agora em vírgula 
    
    # 2. ADICIONA A COLUNA COM O NOME DO ARQUIVO
    # os.path.basename(arquivo) pega só o nome final do arquivo, sem as pastas antes
    df_temp['Arquivo'] = os.path.basename(arquivo)
    
    # Adicionar a tabela na nossa lista
    lista_dataframes.append(df_temp)

# 3. Juntar todas as tabelas em uma só
df_final = pd.concat(lista_dataframes, ignore_index=True)

# 4. Mostrar o resultado para conferir a nova coluna 'Arquivo' (que ficará no final à direita)
print(f"Total de arquivos juntados: {len(arquivos)}")
display(df_final.head())


# 5. Salvar o arquivo gigante final
df_final.to_csv(ORIGINAIS_TODOS, index=False)   # consistente em vírgula

Total de arquivos juntados: 43


,No.,Name,Time (ms),Timegap (ms),Gaze X,Gaze Y,Pupil Diameter Left (mm),Pupil Diameter Right (mm),X-offset,Y-offset,Mouse X-coord,Mouse Y-coord,AOI,Left Eye X,Left Eye Y,Right Eye X,Right Eye Y,Detection,Arquivo,Eye-Tracker ID
0,1126,TESTE 44,19679,17,532,306,4.11,4.24,0.0,0.0,512.0,384.0,-,0.0,0.0,0.0,0.0,NaN,T44 CONTROLE_ON.csv,NaN
1,1127,TESTE 44,19696,17,536,291,4.11,4.27,0.0,0.0,512.0,384.0,-,0.0,0.0,0.0,0.0,NaN,T44 CONTROLE_ON.csv,NaN
2,1128,TESTE 44,19713,17,535,285,4.12,4.30,0.0,0.0,512.0,384.0,-,0.0,0.0,0.0,0.0,NaN,T44 CONTROLE_ON.csv,NaN
3,1129,TESTE 44,19729,17,544,285,4.20,4.36,0.0,0.0,512.0,384.0,-,0.0,0.0,0.0,0.0,NaN,T44 CONTROLE_ON.csv,NaN
4,1130,TESTE 44,19746,17,536,287,4.15,4.40,0.0,0.0,512.0,384.0,-,0.0,0.0,0.0,0.0,NaN,T44 CONTROLE_ON.csv,NaN


## 3. Critério de inclusão

Primeiro o mínimo de 50 janelas; depois o critério técnico.

In [7]:
# ── Filtra participantes com janelas insuficientes ───────────────
# Participantes com menos de N_JANELAS_MIN janelas são excluídos do
# CSV exportado. Isso NÃO remove os arquivos individuais em janelas_on/,
# apenas o consolidado usado nas análises downstream.

N_JANELAS_MIN = 50   # número mínimo de janelas para um participante ser válido

import pandas as pd

_df = pd.read_csv(JANELAS_ON_TODOS)

# Conta janelas por participante (coluna 'Arquivo' identifica o sujeito)
_contagem = _df.groupby('Arquivo')['n_face'].count().rename('n_janelas')

_excluidos = _contagem[_contagem < N_JANELAS_MIN]
_incluidos  = _contagem[_contagem >= N_JANELAS_MIN]

print(f"Participantes válidos  : {len(_incluidos)}")
print(f"Participantes excluídos: {len(_excluidos)}")
if not _excluidos.empty:
    print("\nExcluídos (n_janelas < N_JANELAS_MIN):")
    for arq, n in _excluidos.items():
        print(f"  {arq:<40} {n} janelas")

_df_validos = _df[_df['Arquivo'].isin(_incluidos.index)].copy()
_df_validos.to_csv(JANELAS_ON_VALIDOS, index=False)
print(f"\nSalvo: {JANELAS_ON_VALIDOS.name}  ({len(_df_validos)} linhas, {len(_incluidos)} participantes)")


Participantes válidos  : 42
Participantes excluídos: 0

Salvo: janelas_on_validos.csv  (2100 linhas, 42 participantes)


### Critério de inclusão técnico (qualidade do rastreio + integridade do arquivo)

Calculado em `src/qualidade_sujeitos.py` (`avaliar_sujeitos`), com dois critérios selecionáveis em
`CRITERIO` — ambos fixados a priori, cegos ao grupo e à acurácia:

- **`'protocolo'` (análise PRINCIPAL, N=41)** — exclui só gravações fora do protocolo (duração a
  > 1 s da mediana; as janelas ON são ancoradas no fim da gravação, então duração anômala
  desalinha emoção × tempo). Problemas de rastreio/calibração/proveniência ficam como **alerta**.
- **`'qualidade'` (análise de SENSIBILIDADE, N=37)** — exclui rastreio insuficiente (< 50% das
  janelas com ≥ 50% de amostras válidas) e calibração deslocada (centro do olhar > 200 px), no
  máximo `MAX_EXCLUIDOS = 5`.

**Atenção:** na amostra principal, a taxa de rastreio sozinha separa os grupos (TEA rastreia pior);
o artigo deve reportar a análise de sensibilidade junto.

**Olhar fora do rosto (`perc_face`) NÃO é critério de exclusão** — é o próprio fenômeno estudado
(evitação do olhar em TEA). Tabela completa em `data/interim/qualidade_sujeitos.csv`.


In [8]:
# ── Critério de inclusão técnico (src/qualidade_sujeitos.py) ─────
import sys
sys.path.insert(0, str(RAIZ / 'src'))
from qualidade_sujeitos import avaliar_sujeitos

QUALIDADE_CSV = DIR_INTERIM / "qualidade_sujeitos.csv"

# Calcula sobre TODOS (não sobre os válidos) para a tabela ficar completa e a célula ser idempotente
_qual = avaliar_sujeitos(JANELAS_ON_TODOS, PASTA_FILTRADOS, PASTA_CSV, PASTA_XLSX)
_qual.to_csv(QUALIDADE_CSV, index=False)
display(_qual[['Arquivo', 'Grupo', 'taxa_rastreio', 'perc_jan_validas', 'perc_face',
               'desvio_centro_px', 'fonte', 'concordancia_export', 'duracao_s',
               'EXCLUIR', 'motivo', 'alerta']].round(1))

_excl_qual = _qual.loc[_qual['EXCLUIR'], 'Arquivo']
print(f"\nExcluídos por critério técnico: {len(_excl_qual)}")
for _, r in _qual[_qual['EXCLUIR']].iterrows():
    print(f"  {r['Arquivo']:<28} {r['Grupo']:<9} {r['motivo']}")
for _, r in _qual[~_qual['EXCLUIR'] & (_qual['alerta'] != '')].iterrows():
    print(f"  (mantido, alerta) {r['Arquivo']:<28} {r['Grupo']:<9} {r['alerta']}")

_df_validos = _df[_df['Arquivo'].isin(_incluidos.index) & ~_df['Arquivo'].isin(_excl_qual)].copy()
_df_validos.to_csv(JANELAS_ON_VALIDOS, index=False)
print(f"\nSalvo: {JANELAS_ON_VALIDOS.name}  ({len(_df_validos)} linhas, "
      f"{_df_validos['Arquivo'].nunique()} participantes)")

,Arquivo,Grupo,taxa_rastreio,perc_jan_validas,perc_face,desvio_centro_px,fonte,concordancia_export,duracao_s,EXCLUIR,motivo,alerta
0,T42 CONTROLE.csv,Controle,89.6,98.0,78.0,10.0,reexportado_consistente,100.0,206.0,True,duração fora do protocolo,duração fora do protocolo
1,T28 TEA GRAVE.csv,TEA,2.9,0.0,0.9,109.4,original,100.0,203.9,False,,rastreio insuficiente
2,T17 TEA LEVE.csv,TEA,13.6,12.0,10.0,15.3,original,100.0,203.8,False,,rastreio insuficiente
3,T21 TEA MODERADO.csv,TEA,26.8,24.0,24.4,31.6,original,100.0,203.8,False,,rastreio insuficiente
4,T 24 TEA LEVE.csv,TEA,44.3,34.0,28.3,84.5,original,100.0,203.8,False,,rastreio insuficiente
5,T41 CONTROLE.csv,Controle,66.7,70.0,60.7,9.8,original,100.0,203.9,False,,
6,T30 TEA GRAVE.csv,TEA,66.7,78.0,54.3,32.5,original,100.0,203.8,False,,
7,T32 TEA LEVE.csv,TEA,74.0,78.0,65.8,22.8,original,100.0,203.7,False,,
8,T20 TEA MODERADO.csv,TEA,78.1,92.0,68.2,37.5,original,100.0,203.9,False,,
9,T25 TEA LEVE.csv,TEA,81.8,94.0,73.1,28.8,original,100.0,203.8,False,,



Excluídos por critério técnico: 1
  T42 CONTROLE.csv             Controle  duração fora do protocolo
  (mantido, alerta) T28 TEA GRAVE.csv            TEA       rastreio insuficiente
  (mantido, alerta) T17 TEA LEVE.csv             TEA       rastreio insuficiente
  (mantido, alerta) T21 TEA MODERADO.csv         TEA       rastreio insuficiente
  (mantido, alerta) T 24 TEA LEVE.csv            TEA       rastreio insuficiente
  (mantido, alerta) T15 TEA MODERADO.csv         TEA       CSV diverge do export original
  (mantido, alerta) T12 CONTROLE.csv             Controle  calibração deslocada; CSV diverge do export original

Salvo: janelas_on_validos.csv  (2050 linhas, 41 participantes)


## 4. Gaze com emoção

Carimba em cada amostra a emoção da janela em que ela cai. Saída: `gaze_com_emocoes.csv`.

In [9]:
# ── Carregar dados ──────────────────────────────────────────────
df_gaze  = pd.read_csv(ORIGINAIS_TODOS, low_memory=False)
df_gaze.columns = df_gaze.columns.str.strip()
df_jan   = pd.read_csv(JANELAS_ON_VALIDOS)   # janelas detectadas por vídeo (com emoção)

# ── O RESGATE DA COLUNA 'ARQUIVO' ───────────────────────────────
# Procura qualquer coluna que tenha a palavra 'Arquivo' no nome (ex: 'Detection;Arquivo;No.')
colunas_mutantes = [col for col in df_gaze.columns if 'Arquivo' in col]

if colunas_mutantes:
    coluna_mutante = colunas_mutantes[0]
    
    # Só precisamos tratar se ela estiver grudada com outras coisas
    if coluna_mutante != 'Arquivo':
        # Quebra o nome pelo ';' para saber a posição exata da palavra 'Arquivo'
        partes_do_nome = coluna_mutante.split(';')
        posicao_arquivo = partes_do_nome.index('Arquivo')
        
        # Extrai apenas o pedaço de texto daquela posição específica
        df_gaze['Arquivo'] = df_gaze[coluna_mutante].str.split(';').str[posicao_arquivo]
        
        # Remove a coluna mutante gigante para manter o DataFrame limpo
        df_gaze = df_gaze.drop(columns=[coluna_mutante])

df_gaze['ID_Sujeito']   = df_gaze['Arquivo'].str.extract(r'^(T\s*\d+)')
df_gaze['ID_Sujeito']   = df_gaze['ID_Sujeito'].str.replace(' ', '', regex=False)  # 'T 24' -> 'T24'
df_gaze['Grupo']        = df_gaze['Arquivo'].str.replace('_ON.csv', '', regex=False).str.extract(r'T\s*\d+\s+(.*)')
df_gaze['Grupo']        = df_gaze['Grupo'].str.strip()
df_gaze['Arquivo_base'] = df_gaze['Arquivo'].str.replace('_ON.csv', '.csv', regex=False)
df_gaze['Time (ms)']    = pd.to_numeric(df_gaze['Time (ms)'], errors='coerce')
df_gaze = df_gaze.dropna(subset=['Arquivo_base', 'Time (ms)'])

# ── Intervalos ON (com emoção) vindos das janelas detectadas ───
# Cada linha de janelas_on_todos.csv já é um intervalo ON com sua emoção.
# O nome em 'Arquivo' lá é o do CSV de dados (ex: 'T16 TEA LEVE.csv'),
# que casa com 'Arquivo_base' do gaze.
df_on = df_jan.rename(columns={
    'Arquivo': 'Arquivo_base',
    'on_inicio_ms': 'inicio_ms',
    'on_fim_ms': 'fim_ms',
    'emocao': 'emoção',
})[['Arquivo_base', 'inicio_ms', 'fim_ms', 'emoção']].dropna()

# ── Join por intervalo de tempo ─────────────────────────────────
# Para cada linha de gaze, verifica em qual intervalo da auditoria
# ela se encaixa e stampa a emoção correspondente
registros = []

for arq, grupo_gaze in df_gaze.groupby('Arquivo_base'):
    intervalos = df_on[df_on['Arquivo_base'] == arq]
    if intervalos.empty:
        continue

    grupo_gaze = grupo_gaze.copy()
    grupo_gaze['emocao'] = 'off'

    for _, row in intervalos.iterrows():
        mask = (grupo_gaze['Time (ms)'] >= row['inicio_ms']) & \
               (grupo_gaze['Time (ms)'] <  row['fim_ms'])
        grupo_gaze.loc[mask, 'emocao'] = row['emoção']

    registros.append(grupo_gaze)

df_final = pd.concat(registros, ignore_index=True)

# ── Limpar e renomear colunas ───────────────────────────────────
df_final = df_final.drop(columns=['Arquivo_base'])
df_final = df_final.rename(columns={
    'No.'                       : 'No',
    'Time (ms)'                 : 'Time_ms',
    'Timegap (ms)'              : 'Timegap_ms',
    'Gaze X'                    : 'Gaze_X',
    'Gaze Y'                    : 'Gaze_Y',
    'Pupil Diameter Left (mm)'  : 'Pupil_Left_mm',
    'Pupil Diameter Right (mm)' : 'Pupil_Right_mm',
})

cols = ['ID_Sujeito', 'Grupo', 'Arquivo', 'emocao',
        'Time_ms', 'Timegap_ms', 'Gaze_X', 'Gaze_Y',
        'Pupil_Left_mm', 'Pupil_Right_mm', 'AOI', 'Name', 'No']
df_final = df_final[cols]

# ── Salvar ──────────────────────────────────────────────────────
df_final.to_csv(GAZE_COM_EMOCOES, index=False)
print(f"Shape: {df_final.shape}")
print(df_final['emocao'].value_counts())

Shape: (297158, 13)
emocao
neutro    99996
feliz     99729
raiva     97433
Name: count, dtype: int64
